In [ ]:
import os
import random

path = "../../data/"
limit = 100
regions = [f for f in os.listdir(path) if (os.path.isdir(os.path.join(path, f)) and f != "study areas shp")]

regions_dict = dict()

random.seed(42)

for region in regions:
    dataset_dir = path + region
    image_dir = os.path.join(dataset_dir, "img")
    img_list = os.listdir(image_dir)

    all_images = sorted(os.path.join(image_dir, f) for f in img_list)

    if len(all_images) > limit:
        all_images = random.sample(all_images, limit)

    regions_dict[region] = all_images

{'Hokkaido Iburi-Tobu': ['../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido1371.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0252.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0057.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0596.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0534.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0489.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0309.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0231.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido1454.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido1168.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0200.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido1269.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0910.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0077.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0071.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0213.tif', '../../data/Hokkaido Iburi-Tobu\\img\\Hokkaido0479.tif', '../..

In [5]:
import numpy as np
import pandas as pd
import tifffile as tif
import imageio.v3 as iio
from scipy.ndimage import sobel

region_features = []

for region, img_paths in regions_dict.items():

    lfhf_vals = []
    centroid_vals = []
    entropy_vals = []
    edge_vals = []
    grad_vals = []

    for img_path in img_paths:

        # --- LOAD IMAGE ---
        # Uses tifffile for .tif and imageio for others
        try:
            if img_path.lower().endswith(".tif"):
                img = tif.imread(img_path)
            else:
                img = iio.imread(img_path)
        except Exception as e:
            print("Failed to load:", img_path, e)
            continue

        # --- ENSURE GRAYSCALE ---
        if img.ndim == 3:          # RGB or multispectral
            gray = img.mean(axis=2).astype(float)
        elif img.ndim == 2:        # already grayscale
            gray = img.astype(float)
        else:
            print("Odd shape", img.shape, "skipping")
            continue

        # --- FFT ---
        F = np.fft.fft2(gray)
        mag = np.abs(F)

        # --- LF/HF ratio ---
        h, w = mag.shape
        cy, cx = h//2, w//2
        r = min(h, w)//8
        yy, xx = np.ogrid[:h, :w]
        mask = (yy - cy)**2 + (xx - cx)**2 <= r*r

        lf = mag[mask].mean()
        hf = mag[~mask].mean()
        lfhf_vals.append(lf / (hf + 1e-9))

        # --- Spectral centroid ---
        fy = np.fft.fftfreq(h)
        fx = np.fft.fftfreq(w)
        FY, FX = np.meshgrid(fy, fx, indexing="ij")
        centroid_vals.append(np.sum(mag * np.sqrt(FX**2 + FY**2)) / (mag.sum() + 1e-9))

        # --- Entropy ---
        p = mag.ravel()
        p = p / (p.sum() + 1e-12)
        entropy_vals.append(-(p * np.log(p + 1e-12)).sum())

        # --- Gradient & edges ---
        sx = sobel(gray, axis=0)
        sy = sobel(gray, axis=1)
        edges = np.hypot(sx, sy)

        edge_vals.append((edges > edges.mean()).mean())
        grad_vals.append(edges.mean())

    # SAVE REGION AVERAGES
    region_features.append({
        "region": region,
        "lfhf": np.mean(lfhf_vals),
        "centroid": np.mean(centroid_vals),
        "entropy": np.mean(entropy_vals),
        "edge": np.mean(edge_vals),
        "grad": np.mean(grad_vals)
    })

# CONVERT TO DF
feat_df = pd.DataFrame(region_features)
feat_df.to_csv("region_features.csv", index=False)

print("Saved region_features.csv with", len(feat_df), "regions")

Saved region_features.csv with 16 regions
